## STEP 2. 실제 스키마·PK·FK를 먼저 검증

In [9]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("현재 위치:", Path.cwd())
print("프로젝트 루트:", PROJECT_ROOT)
print("src 폴더 존재:", (PROJECT_ROOT / "src").exists())
print(
    "llm_code_validation.py 존재:",
    (PROJECT_ROOT / "src" / "llm_code_validation.py").exists()
)

현재 위치: c:\dev\kant-axagent-study\llm-data-analysis-course\chapter12
프로젝트 루트: c:\dev\kant-axagent-study\llm-data-analysis-course
src 폴더 존재: True
llm_code_validation.py 존재: False


In [11]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)

print("현재 작업 위치:", Path.cwd())

from src.llm_code_validation import (

    assert_validation_ready,

    build_dataset_inventory,

    load_validation_data,

    validate_primary_keys,

    validate_relationship_keys,

    validate_required_columns,

)

datasets = load_validation_data("data/processed")

inventory = build_dataset_inventory(datasets)

required_column_check = validate_required_columns(datasets)

primary_key_check = validate_primary_keys(datasets)

relationship_check = validate_relationship_keys(datasets)

display(inventory)

display(required_column_check)

display(primary_key_check)

display(relationship_check)

assert_validation_ready(

    required_column_check,

    primary_key_check,

    relationship_check,

)

현재 작업 위치: c:\dev\kant-axagent-study\llm-data-analysis-course


,dataset,exists,rows,columns,column_list,missing_values,duplicated_rows
0,customers,True,150,6,"customer_id, name, gender, age, city, signup_date",0,0
1,products,True,100,4,"product_id, product_name, category, price",0,0
2,orders,True,300,7,"order_id, customer_id, order_date, payment_met...",0,0
3,order_items,True,764,6,"order_item_id, order_id, product_id, quantity,...",0,0


,dataset,column,dataset_exists,exists,status
0,customers,age,True,True,PASS
1,customers,city,True,True,PASS
2,customers,customer_id,True,True,PASS
3,customers,gender,True,True,PASS
4,products,category,True,True,PASS
5,products,price,True,True,PASS
6,products,product_id,True,True,PASS
7,products,product_name,True,True,PASS
8,orders,customer_id,True,True,PASS
9,orders,order_date,True,True,PASS


,dataset,primary_key,key_exists,missing_key_count,duplicated_key_count,status
0,customers,customer_id,True,0,0,PASS
1,products,product_id,True,0,0,PASS
2,orders,order_id,True,0,0,PASS
3,order_items,order_item_id,True,0,0,PASS


,purpose,left_dataset,right_dataset,key,left_has_key,right_has_key,missing_left_key_count,duplicated_parent_key_count,invalid_reference_count,status
0,상품 정보와 주문 상세 연결,order_items,products,product_id,True,True,0,0,0,PASS
1,주문 정보와 주문 상세 연결,order_items,orders,order_id,True,True,0,0,0,PASS
2,주문 정보와 고객 정보 연결,orders,customers,customer_id,True,True,0,0,0,PASS


### STEP 02 결과 요약

- customers, products, orders, order_items 4개 데이터셋이 모두 정상적으로 존재하는 것을 확인했다.
- 데이터 크기는 customers 150행, products 100행, orders 300행, order_items 764행이었다.
- 네 데이터셋 모두 결측값이 0개이고 중복 행도 0개였다.
- 분석에 필요한 필수 컬럼이 모두 존재했으며 필수 컬럼 검증 결과는 모두 PASS였다.
- `order_items.order_item_id`를 포함하여 Chapter 12에서 요구하는 데이터 구조를 확인했다.
- 마지막 `assert_validation_ready()` 실행에서 오류가 발생하지 않아 STEP 02 검증을 통과했다.
- 따라서 현재 processed 데이터는 다음 단계인 merge 및 completed 주문 집계 검증에 사용할 수 있다고 판단했다.

## STEP 3. merge와 completed 집계의 Business Rule 검증

In [12]:
from src.llm_code_validation import safe_category_sales

category_sales, category_validation = safe_category_sales(
    order_items=datasets["order_items"],
    products=datasets["products"],
    orders=datasets["orders"],
)

display(category_sales)
display(category_validation)

,category,total_quantity,total_sales,sales_ratio
0,스포츠,295,31743000,21.31
1,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
3,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
5,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,check_item,value,status
0,order_merge_row_count,True,PASS
1,order_merge_unmatched,0,PASS
2,completed_detail_rows,474,PASS
3,included_status,completed,PASS
4,product_merge_row_count,True,PASS
5,product_merge_unmatched,0,PASS
6,category_missing,0,PASS
7,completed_source_total,148990000.0,PASS
8,category_grouped_total,148990000.0,PASS
9,category_total_difference,0.0,PASS


### STEP 03 결과 요약

- completed 상태의 주문 상세 행은 474건이었다.
- 주문 merge 후 행 수가 유지되었고, 주문 미매칭 건수는 0건이었다.
- 상품 merge 후에도 행 수가 유지되었고, 상품 미매칭 건수는 0건이었다.
- category 결측값은 0건이었다.
- completed 주문의 원본 상세 금액 합계는 148,990,000원이었다.
- 카테고리별 집계 금액 합계도 148,990,000원으로 동일했다.
- 두 금액의 차이는 0원으로 확인되었다.
- 카테고리별 매출 비율의 합계는 100%로 확인되었다.
- 모든 검증 항목이 PASS였으므로, 카테고리별 매출 집계 과정에서 merge 누락이나 금액 손실은 발견되지 않았다.

## STEP 4. Generated Code를 실행하지 않고 AST Static Scan

In [13]:
from src.llm_code_validation import (
    DEFAULT_STATIC_SCAN_EXAMPLE,
    scan_generated_code,
)

print(DEFAULT_STATIC_SCAN_EXAMPLE)

import requests

response = requests.get("https://example.com/data.csv")
open("download.csv", "wb").write(response.content)


In [14]:
static_scan = scan_generated_code(
    DEFAULT_STATIC_SCAN_EXAMPLE
)

display(static_scan)

,severity,category,line,detail
0,review,import,1,외부 작업 가능 모듈 import: requests
1,high,network,3,외부 URL 사용: requests.get
2,high,operation,3,외부 네트워크 요청: requests.get
3,high,file_write,4,"파일 쓰기 모드: open(..., 'wb')"


### STEP 04 결과 요약

- 정적 스캔 결과 총 4개의 위험 또는 검토 항목이 탐지되었다.
- `requests` 모듈 import는 외부 작업 가능성이 있어 `review`로 탐지되었다.
- `requests.get()`은 외부 URL에 접근하는 네트워크 요청으로 판단되어 `high` 위험으로 탐지되었다.
- 같은 `requests.get()` 호출은 실제 외부 네트워크 작업이라는 점에서도 `high` 위험으로 탐지되었다.
- `open(..., "wb")`는 파일을 바이너리 쓰기 모드로 저장하는 동작이어서 `high` 위험으로 탐지되었다.
- 따라서 이 예제 코드는 그대로 실행하면 안 되고, 현재 상태에서는 BLOCK 대상으로 판단할 수 있다.

## STEP 5. 회귀와 분류의 Feature Contract를 따로 검토

In [15]:
from src.llm_code_validation import (
    build_feature_audit,
    build_leakage_review_table,
    validate_feature_list,
)

leakage_contract = build_leakage_review_table()

display(leakage_contract)

,problem,prediction_time,target,forbidden_examples,split_rule
0,regression,주문 상세 기반 target 재료를 아직 사용할 수 없는 시점,order_total,"avg_unit_price, customer_id, item_count, line_...",날짜 순서 split + train 내부 TimeSeriesSplit selecti...
1,classification,주문 생성 직후,"completed=0, cancelled=1; refunded/other 제외","cancel_reason, cancelled_at, customer_id, is_c...",교육용 stratified train/validation/test; model/th...


In [16]:
classification_features = [
    "payment_method",
    "item_count",
    "total_quantity",
    "order_amount",
    "age",
    "city",
]

validate_feature_list(
    classification_features,
    problem="classification",
)

display(
    build_feature_audit(
        classification_features,
        problem="classification",
    )
)

,problem,feature,status,review_note
0,classification,payment_method,REVIEW,예측 시점 가용성을 사람이 확인
1,classification,item_count,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
2,classification,total_quantity,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
3,classification,order_amount,REVIEW,주문 생성 시 상품 구성·수량·금액이 확정되어 있다는 교육용 가정 확인
4,classification,age,REVIEW,예측 시점 가용성을 사람이 확인
5,classification,city,REVIEW,예측 시점 가용성을 사람이 확인


### STEP 05 결과 요약

- 회귀와 분류는 서로 다른 예측 시점과 Feature Contract를 사용해야 함을 확인했다.
- 분류 문제의 예측 시점은 주문 생성 직후로 정의되어 있다.
- payment_method, item_count, total_quantity, order_amount, age, city를 검토한 결과 모두 REVIEW로 분류되었다.
- REVIEW는 해당 Feature가 금지되었다는 뜻이 아니라, 예측 시점에 실제로 사용할 수 있는 정보인지 사람이 확인해야 한다는 뜻이다.
- item_count, total_quantity, order_amount는 주문 생성 시 상품 구성·수량·금액이 이미 확정되어 있다는 가정하에서 사용할 수 있다.
- 따라서 Feature Leakage는 단순히 컬럼 이름만으로 판단하는 것이 아니라, 예측 시점에 해당 정보를 알 수 있었는지를 기준으로 판단해야 한다.

## STEP 6. Sandbox와 Package 검토

In [19]:
from src.llm_code_validation_policy import run_llm_code_validation

result = run_llm_code_validation(
    processed_dir="data/processed",
    report_dir="reports",
)

display(result["outputs"]["sandbox_checklist"])
display(result["outputs"]["package_review"])

,check_item,status,evidence
0,운영 데이터가 아닌 복사한 소량 샘플을 사용하는가?,REVIEW,
1,입력 데이터는 가능하면 read-only로 제공되는가?,REVIEW,
2,API Key·클라우드 credential·DB 비밀번호가 없는 환경인가?,REVIEW,
3,실행 환경은 작업 후 폐기 가능한 disposable 환경인가?,REVIEW,
4,쓰기 허용 경로가 별도 output 폴더 등 allowlist로 제한되는가?,REVIEW,
5,불필요한 네트워크 접근을 기본 차단했는가?,REVIEW,
6,CPU·메모리·실행 시간·프로세스 수·디스크 사용량 제한이 있는가?,REVIEW,
7,실행 전 저장소와 대상 폴더의 baseline 상태를 기록했는가?,REVIEW,
8,"실행 시작·종료 시각, exit code, timeout 여부를 기록할 수 있는가?",REVIEW,
9,실행 후 생성·수정 파일 목록과 허용 경로 밖 변경을 비교할 수 있는가?,REVIEW,


,package,requested_version,purpose,package_needed,official_source_verified,name_typo_or_typosquatting_checked,exact_version_pinned,python_compatibility_checked,transitive_dependencies_reviewed,install_script_reviewed,isolated_environment_planned,requirements_or_lock_recorded,organization_policy_checked,decision
0,,,,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,REVIEW,DO_NOT_INSTALL_UNTIL_REVIEWED


### STEP 06 결과 요약

- Sandbox 실행 체크리스트의 11개 항목이 모두 REVIEW로 확인되었다.
- 이는 실행 환경의 안전 조건이 자동으로 승인된 것이 아니라, 사람이 실제 Evidence를 확인해야 한다는 의미이다.
- 샘플 데이터 사용 여부, read-only 입력, credential 제거, 네트워크 차단, 자원 제한, 실행 전후 변경 기록 등을 추가로 확인해야 한다.
- Package 검토 결과도 모든 항목이 REVIEW 상태였다.
- 패키지 설치 최종 결정은 `DO_NOT_INSTALL_UNTIL_REVIEWED`로 확인되었다.
- 따라서 새 패키지는 공식 출처, 정확한 버전, Python 호환성, 의존성, 설치 스크립트 등을 검토하기 전에는 설치하지 않아야 한다.
- 현재 단계에서는 Sandbox와 Package 모두 사람의 추가 검토가 필요한 상태이다.

## STEP 7. 전체 Evidence와 Execution Gate 확인

In [20]:
display(result["outputs"]["execution_gate"])

,gate,status,meaning
0,schema_and_keys,PASS,필수 구조·PK·관계
1,aggregate_validation,PASS,completed 범위·병합·총합
2,ml_leakage,REVIEW,문제별 prediction time·forbidden feature·split/se...
3,static_scan,BLOCKED,차단 심각도 탐지: high
4,sandbox_and_package,REVIEW,격리 환경·쓰기 범위·네트워크·리소스 제한·공급망을 사람이 확인
5,human_approval,REVIEW,자동 Evidence와 별도로 명시적 실행 승인 필요
6,execution_decision,DO_NOT_EXECUTE,자동 승인하지 않음


### STEP 07 결과 요약

- schema_and_keys는 PASS로 확인되었다.
- aggregate_validation도 PASS로 확인되어 데이터 구조와 completed 집계 과정에는 문제가 없었다.
- ml_leakage는 REVIEW 상태로, Feature의 예측 시점 가용성을 사람이 추가 확인해야 한다.
- static_scan에서는 high 위험 항목이 탐지되어 BLOCKED로 판정되었다.
- sandbox_and_package와 human_approval도 REVIEW 상태로, 실행 환경과 사람의 명시적 승인이 필요하다.
- 최종 execution_decision은 DO_NOT_EXECUTE로 확인되었다.
- 이는 자동 검증이 실패한 것이 아니라, 위험 코드가 포함된 상태에서 실행을 차단한 정상적인 결과이다.

## STEP 08. 사람의 실행 전 판단

**판단: REVISE**

#### 판단 근거

- 데이터 구조, PK/FK, completed 집계 검증은 PASS였다.
- 하지만 Static Scan에서 외부 네트워크 요청과 파일 쓰기 동작이 high 위험으로 탐지되었다.
- Sandbox와 Package 검토도 아직 REVIEW 상태이므로 실행 환경의 안전성이 충분히 확인되지 않았다.
- 따라서 현재 코드를 바로 실행하지 않고 위험 동작을 제거하거나 제한한 뒤 다시 검토해야 한다.

#### 수정 방향

- 불필요한 외부 네트워크 요청 제거
- 파일 쓰기 경로를 승인된 output 폴더로 제한
- Sandbox 조건 확인
- 수정 후 Static Scan과 Execution Gate 재검토

## STEP 09. 제한 실행 여부

- STEP 08의 사람 판단은 REVISE였다.
- Execution Gate의 최종 결정은 DO_NOT_EXECUTE였다.
- Static Scan에서 high 위험 항목이 탐지되어 BLOCKED 상태였다.
- 따라서 현재 코드는 실행하지 않았다.
- 위험 동작을 수정하고 Static Scan과 Execution Gate를 다시 통과한 뒤,
  사람이 APPROVE한 경우에만 제한 환경에서 실행할 예정이다.

## STEP 10. 실행 후 검증

- STEP 09에서 현재 코드를 실행하지 않았기 때문에 Post-execution Validation은 수행하지 않았다.
- Static Scan에서 high 위험 항목이 탐지되었고 Execution Gate가 DO_NOT_EXECUTE였으므로 실행 중단이 적절했다.
- 따라서 생성 파일, 수정 파일, 삭제 파일, exit code, timeout 여부 등의 실행 후 Evidence는 발생하지 않았다.
- 위험 동작을 수정하고 사람이 APPROVE한 코드가 실제로 제한 실행된 이후에 Post-execution Validation을 수행해야 한다.

## STEP 11. 오류를 LLM에게 다시 물을 때도 최소 Context만 공유

In [21]:
from src.llm_code_validation import build_error_fix_prompt_template

print(build_error_fix_prompt_template())

# 오류 수정 요청 — 최소·비식별 Context만 사용

분석 목적:
- [계산/예측하려는 내용을 한 문장으로 작성]

실제 필요한 데이터 구조:
- [데이터셋과 필요한 컬럼만 작성]
- 원본 고객/거래 행, 개인정보, API Key, 내부 URL, 사용자명, 절대 경로는 제공하지 않음

최소 재현 코드:
```python
[오류를 재현하는 최소 코드만]
```

민감정보를 제거한 오류 또는 검증 결과:
```text
[예외 유형, 필요한 메시지, 행 수, 미매칭 수, 총합 차이 등]
```

요청:
1. 실행 오류와 분석 논리 오류를 구분해서 설명해 주세요.
2. 실제 제공한 컬럼명만 사용하는 최소 수정안을 제안해 주세요.
3. merge validate, 전후 행 수, 미매칭, 총합 대조를 포함해 주세요.
4. 원인을 확인하지 못한 부분은 추측하지 말고 확인 방법을 제시해 주세요.
5. 파일 삭제·덮어쓰기, 외부 통신, OS 명령, package 설치 코드를 추가하지 마세요.
6. 수정 코드와 함께 사람이 다시 검증할 항목을 표로 제시해 주세요.



## STEP 11. 오류를 LLM에게 다시 물을 때도 최소 Context만 공유


### STEP 11 결과 요약

- 오류 해결을 LLM에게 요청할 때 전체 데이터와 전체 로그를 공유하지 않고 최소 Context만 제공하는 Prompt Template을 확인했다.
- 분석 목적, 필요한 데이터셋과 컬럼, 최소 재현 코드, 비식별화된 오류 메시지와 검증 수치만 공유하도록 구성되어 있다.
- 원본 고객/거래 행, 개인정보, API Key, Token, DB password, 내부 URL, 사용자명, 절대 경로 등은 공유하지 않도록 되어 있다.
- 오류 수정 요청 시 실행 오류와 분석 논리 오류를 구분하도록 요구한다.
- 실제 존재하는 컬럼만 사용하도록 하고, merge 전후 행 수, 미매칭 수, 총합 검증을 다시 수행하도록 요구한다.
- 원인이 확인되지 않은 부분은 추측하지 않고 추가 확인 방법을 제시하도록 한다.
- 파일 삭제·덮어쓰기, 외부 통신, OS 명령, package 설치 같은 새로운 위험 동작을 추가하지 않도록 제한한다.

- Human Decision: REVISE
- 이유: 데이터 구조와 집계 검증은 통과했지만, Static Scan에서 외부 네트워크 요청과 파일 쓰기 관련 high 위험 항목이 탐지되었다.
- 현재 상태에서는 실행하지 않고, 위험 동작을 수정한 뒤 다시 Static Scan과 Execution Gate를 검토해야 한다.

## STEP 12. Evidence 파일과 최종 신뢰 범위 확인


In [22]:
from pathlib import Path

report_dir = Path("reports")

for file in sorted(report_dir.glob("ch12_*")):
    print(file.name)

ch12_category_sales_validated.csv
ch12_category_sales_validation.csv
ch12_code_validation_summary.md
ch12_dataset_inventory.csv
ch12_error_fix_prompt_template.md
ch12_execution_gate.csv
ch12_generated_code_static_scan.csv
ch12_human_revision_log.csv
ch12_llm_code_review_checklist.csv
ch12_ml_leakage_review.csv
ch12_monthly_sales_validated.csv
ch12_monthly_sales_validation.csv
ch12_package_install_review.csv
ch12_primary_key_check.csv
ch12_relationship_key_check.csv
ch12_required_column_check.csv
ch12_sandbox_execution_checklist.csv


In [23]:
print(result["outputs"]["validation_summary"])

# Chapter 12 LLM 분석 코드 검증 요약

## 핵심 원칙

- 생성 코드는 검토되지 않은 초안입니다.
- 코드가 실행됨 ≠ 분석이 올바름.
- 정적 스캔 0건 ≠ 코드가 안전함.
- 자동 검증 PASS ≠ 실행 승인. 사람 승인과 제한된 실행 환경이 별도로 필요합니다.

## 1. 데이터셋 인벤토리
```text
    dataset  exists  rows  columns                                                                                   column_list  missing_values  duplicated_rows
  customers    True   150        6                                             customer_id, name, gender, age, city, signup_date               0                0
   products    True   100        4                                                     product_id, product_name, category, price               0                0
     orders    True   300        7 order_id, customer_id, order_date, payment_method, order_status, order_month, order_dayofweek               0                0
order_items    True   764        6                         order_item_id, order_id, product_id, quantity, unit_price, line_total               0                0
```

##

## 12. 최종 신뢰 판단

**추가 검증 후 사용 가능**

신뢰 가능한 범위:
- processed 데이터 구조
- 필수 컬럼
- PK/FK 관계
- completed 주문 범위
- merge 전후 행 수와 미매칭
- 카테고리별·월별 금액 총합

추가 검토가 필요한 범위:
- Feature의 실제 prediction time 가용성
- 외부 네트워크 접근
- 파일 쓰기 동작
- Sandbox 환경
- Package 설치
- 사람의 최종 실행 승인